In [2]:
# making a function to append a value to a list 
def append_val(val, lst=None) :
    if lst ==None :
        lst = []
    lst.append(val)

    return lst 

In [3]:
append_val(45)

[45]

In [4]:
append_val(90)

[90]

In [7]:
# impliment a bsiac queue... 
class TaskQueue:
    def __init__(self):
        self._tasks = [] # list of dicts 

    def add_task(self,task_id: str,priority: int, payload: dict) -> None:
        self._tasks.append({ "id": task_id,
                             "priority": priority,
                             "payload": payload,
                             "status": "pending"})

    def get_pending(self)-> list:
        return [t for t in self._tasks if t["status"] == "pending"]

    def get_by_id(self, task_id: str)-> dict:

        for t in self._tasks:
            if t["id"]== task_id:

                return t

        return None 
    

In [9]:
# 1. Instantiate the queue
queue = TaskQueue()

# 2. Add some tasks using your add_task method
# Arguments: task_id, priority, payload (a dict)
queue.add_task("task_001", 10, {"action": "render_video", "frame": 45})
queue.add_task("task_002", 5, {"action": "send_email", "to": "user@example.com"})

print("--- Testing get_pending() ---")
# 3. Get all pending tasks
pending_tasks = queue.get_pending()
print(f"Found {len(pending_tasks)} pending tasks:")
for task in pending_tasks:
    print(task)

print("\n--- Testing get_by_id() ---")
# 4. Find a specific task by its ID
target_id = "task_002"
found_task = queue.get_by_id(target_id)
print(f"Result for ID '{target_id}':", found_task)

--- Testing get_pending() ---
Found 2 pending tasks:
{'id': 'task_001', 'priority': 10, 'payload': {'action': 'render_video', 'frame': 45}, 'status': 'pending'}
{'id': 'task_002', 'priority': 5, 'payload': {'action': 'send_email', 'to': 'user@example.com'}, 'status': 'pending'}

--- Testing get_by_id() ---
Result for ID 'task_002': {'id': 'task_002', 'priority': 5, 'payload': {'action': 'send_email', 'to': 'user@example.com'}, 'status': 'pending'}


### leve 2 Priority Processing 

In [11]:
def pop_higghest_priority(self)-> dict : 
    pending= [t for t in self_tasks if t["status"] == "pending"]

    if not pending:
        return None 
    task =max(pending, key =lambda t : t["priority"])
    task["status"] = "in_progress"
    return task 
def complete_task(self,task_id: str, result: any) -> None:

    task = self.get_by_id(task_id)
    if task :
        task["status"] = "done"
        task["result"] = result 

def get_stats(self) -> dict:
    from collections import Counter
    counts = Counter(t["status"] for t in self._tasks)
    return dict(counts)



## Level 3 Async Handler 

In [13]:
import asyncio
from asyncio import Queue ## imp

class AsyncTaskQueue:
    def __init__(self, num_workers: int = 3):
        self._q = Queue()
        self._results = {}
        self._lock = asyncio.Lock()
        self._num_workers = num_workers

    async def add_task(self, task_id: str, payload: dict):
        await self._q.put({"id": task_id, "payload": payload})

    async def _worker(self, worker_id: int, handler):
        while True:
            task = await self._q.get()
            if task is None:
                break
            result = await handler(task["payload"])
            async with self._lock:
                self._results[task["id"]] = result
            self._q.task_done()

    async def run(self, tasks: list, handler):
        for task in tasks:
            await self._q.put(task)
        workers = [asyncio.create_task(self._worker(i, handler))
                   for i in range(self._num_workers)]
        await self._q.join()
        for _ in workers:
            await self._q.put(None)
        await asyncio.gather(*workers)
        return self._results




## Level 1 — Basic Queue

This level builds a synchronous repository using standard Python data structures: a list filled with dictionaries.

```python
class TaskQueue:

```

* **`class TaskQueue:`** Declares a new object blueprint named `TaskQueue`.

```python
    def __init__(self):
        self._tasks = []  # list of dicts

```

* **`def __init__(self):`** The initializer (constructor) method. It runs automatically whenever you create an instance of this class.
* **`self._tasks = []`** Creates an empty instance variable list. The single underscore (`_`) is a naming convention signaling that this variable is intended to be private inside the class.

```python
    def add_task(self, task_id: str, priority: int, payload: dict) -> None:

```

* **`def add_task(...)`**: Defines a method to insert new tasks.
* **`task_id: str, priority: int, payload: dict`**: Type hints. They specify that `task_id` should be text, `priority` an integer, and `payload` a dictionary holding task-specific data.
* **`-> None`**: A type hint showing that this method modifies the queue in-place and does not return any value.

```python
        self._tasks.append({
            "id": task_id, "priority": priority,
            "payload": payload, "status": "pending"
        })

```

* **`self._tasks.append({...})`**: Appends a dictionary object to the end of the `_tasks` list. The status defaults to `"pending"`.

```python
    def get_pending(self) -> list:
        return [t for t in self._tasks if t["status"] == "pending"]

```

* **`-> list`**: Hints that this method outputs a list.
* **`[t for t in self._tasks if ...]`**: A **List Comprehension**. It loops through every task dictionary `t` inside `self._tasks`, filters for items where `t["status"]` matches `"pending"`, and extracts them into a brand-new list.

```python
    def get_by_id(self, task_id: str) -> dict:
        for t in self._tasks:
            if t["id"] == task_id:
                return t
        return None

```

* **`for t in self._tasks:`**: Iterates step-by-step through the list of task dictionaries.
* **`if t["id"] == task_id:`**: Looks up the value belonging to the `"id"` key inside the current task dictionary and checks if it matches the requested string.
* **`return t`**: Immediately exits the method and returns the matching task dictionary.
* **`return None`**: Executed only if the loop runs to completion without finding a matching ID.

---

## Level 2 — Priority Processing

This level adds custom processing logic to manipulate the status states of the tasks.

```python
    def pop_highest_priority(self) -> dict:

```

* **`-> dict`**: This method locates, modifies, and returns a task dictionary.

```python
        pending = [t for t in self._tasks if t["status"] == "pending"]
        if not pending:
            return None

```

* **`pending = [...]`**: Collects all currently pending tasks using a list comprehension.
* **`if not pending:`**: Evaluates if the `pending` list is completely empty. If it is empty, it returns `None`.

```python
        task = max(pending, key=lambda t: t["priority"])

```

* **`max(pending, ...)`**: Scans the `pending` list to find the maximum element.
* **`key=lambda t: t["priority"]`**: An inline, anonymous function (**lambda**). It instructs `max()` not to compare the dictionaries themselves, but instead to look inside each dictionary `t` and compare them using the integer value stored under the `"priority"` key.

```python
        task["status"] = "in_progress"
        return task

```

* **`task["status"] = "in_progress"`**: Mutates the chosen task's dictionary directly, updating its status.
* **`return task`**: Returns the reference to this updated dictionary.

```python
    def complete_task(self, task_id: str, result: any) -> None:

```

* **`result: any`**: Type hint indicating that the task outcome can be any object type (string, dict, int, etc.).

```python
        task = self.get_by_id(task_id)
        if task:
            task["status"] = "done"
            task["result"] = result

```

* **`task = self.get_by_id(task_id)`**: Reuses the Level 1 method to locate the specific task.
* **`if task:`**: Checks if a dictionary was found (ensures it is not `None`).
* **`task["status"] = "done"`**: Updates the task state.
* **`task["result"] = result`**: Creates a brand new key (`"result"`) inside that task's dictionary to record the incoming data.

```python
    def get_stats(self) -> dict:
        from collections import Counter
        counts = Counter(t["status"] for t in self._tasks)
        return dict(counts)

```

* **`from collections import Counter`**: An inline import of Python's high-performance counting container.
* **`Counter(t["status"] for t in self._tasks)`**: Evaluates a generator expression that pulls out the status string of every task. `Counter` groups them and counts the occurrences (e.g., `{"pending": 2, "done": 1}`).
* **`return dict(counts)`**: Casts the specialized Counter object into a standard Python dictionary and returns it.

---

## Level 3 — Concurrent Workers (asyncio)

This moves away from synchronous code and introduces asynchronous concurrency using non-blocking queues and multiple background worker pools.

```python
import asyncio
from asyncio import Queue

```

* **`import asyncio`**: Imports Python's built-in asynchronous framework.
* **`from asyncio import Queue`**: Imports a specialized asynchronous FIFO (First-In, First-Out) data queue designed for co-routines.

```python
class AsyncTaskQueue:
    def __init__(self, num_workers: int = 3):
        self._q = Queue()
        self._results = {}
        self._lock = asyncio.Lock()
        self._num_workers = num_workers

```

* **`self._q = Queue()`**: Initializes the empty async queue.
* **`self._results = {}`**: An empty dictionary used to store final computation records.
* **`self._lock = asyncio.Lock()`**: Instantiates an async mutual exclusion lock (Mutex). It ensures only one worker changes a resource at a time when explicitly acquired.
* **`num_workers: int = 3`**: A default parameter. If no number is passed, it spins up 3 workers.

```python
    async def add_task(self, task_id: str, payload: dict):
        await self._q.put({"id": task_id, "payload": payload})

```

* **`async def`**: Declares that this function is an asynchronous coroutine.
* **`await self._q.put(...)`**: Places an item into the queue. Because the queue could have a capacity limit (though infinite here), `await` yields control back to the event loop until the insertion completes without freezing the program.

```python
    async def _worker(self, worker_id: int, handler):
        while True:

```

* **`handler`**: A reference to an external async processing function passed in as an argument.
* **`while True:`**: An infinite loop that keeps the background worker alive, continuously waiting for incoming queue jobs.

```python
            task = await self._q.get()

```

* **`await self._q.get()`**: The worker pauses here execution-wise. It hands control back to the event loop until an item is available in `self._q`. Once an item is ready, it unpacks it into `task`.

```python
            if task is None:
                break

```

* **`if task is None:`**: This is a shutdown mechanism often called a **Poison Pill**. If `None` is pulled from the queue, it breaks the infinite loop, terminating this worker.

```python
            result = await handler(task["payload"])

```

* **`await handler(...)`**: Executes the user's custom function on the task data. `await` ensures the worker safely waits for this specific task's computation to completely finish.

```python
            async with self._lock:
                self._results[task["id"]] = result

```

* **`async with self._lock:`**: Requests ownership of the async lock. Once acquired, it safely runs the block underneath.
* **`self._results[...] = result`**: Inserts the computed value into the results dictionary using the unique task ID as the lookup key.

```python
            self._q.task_done()

```

* **`self._q.task_done()`**: Signals to the internal queue tracking mechanism that this specific item has been completely processed.

```python
    async def run(self, tasks: list, handler):

```

* **`run`**: The main orchestration method that schedules the execution of the entire pipeline.

```python
        for task in tasks:
            await self._q.put(task)

```

* Loops through the input list of tasks and pushes every single one of them directly into the async data queue.

```python
        workers = [asyncio.create_task(self._worker(i, handler))
                   for i in range(self._num_workers)]

```

* **`asyncio.create_task(...)`**: Submits the worker coroutines to the background event loop, firing them up immediately as parallel concurrent tasks.
* **`workers = [...]`**: A list comprehension capturing reference handles to these 3 running background worker loops.

```python
        await self._q.join()

```

* **`await self._q.join()`**: Pauses the main program context here until every single task added to the queue has received a corresponding `task_done()` confirmation signal from the workers.

```python
        for _ in workers:
            await self._q.put(None)

```

* Pushes a `None` token (poison pill) into the queue for each worker task to signal that it is time for them to break their internal loops and shut down.

```python
        await asyncio.gather(*workers)
        return self._results

```

* **`await asyncio.gather(*workers)`**: The asterisk (`*`) unpacks the `workers` list into separate arguments. `gather` waits until all workers exit cleanly.
* **`return self._results`**: Returns the populated output data map.

---

## Level 4 — Rate Limiting

This level limits how many operations can happen simultaneously using a traffic control tool called a Semaphore.

```python
    async def _rate_limited_worker(self, sem: asyncio.Semaphore, handler):

```

* **`sem: asyncio.Semaphore`**: An object managing an internal counter. It limits how many async workers can enter a guarded section of code at the exact same time.

```python
        while True:
            task = await self._q.get()
            if task is None:
                break
            async with sem:                    # max N concurrent actual executions
                result = await handler(task["payload"])

```

* **`async with sem:`**: Before running the handler, the worker must acquire a slot from the semaphore. If the maximum allowed number of concurrent slots (e.g., 2) is already active, this worker waits here.
* Once an active handler completes, a slot opens up, and this waiting worker automatically proceeds.

```python
            async with self._lock:
                self._results[task["id"]] = result
            self._q.task_done()

```

* Writes the output to the master dict, and marks the item done exactly like Level 3.

---

## Level 5 — Retries and Error Handling

This worker variant wraps processing within a resilient loop to catch runtime failures and retry them.

```python
    async def _worker_with_retry(self, handler, max_retries: int = 3):
        while True:
            task = await self._q.get()
            if task is None:
                break

```

* Standard task extraction loop setup.

```python
            for attempt in range(max_retries):

```

* **`for attempt in range(max_retries):`**: A retry loop that runs up to `max_retries` times (e.g., indices `0`, `1`, `2`).

```python
                try:
                    result = await handler(task["payload"])
                    async with self._lock:
                        self._results[task["id"]] = {"status": "ok", "result": result}
                    break

```

* **`try:`**: Monitors the block below for runtime crashes or network exceptions.
* **`self._results[...] = {"status": "ok", ...}`**: If successful, it logs a dictionary containing the output data.
* **`break`**: Exits the retry `for` loop immediately, since the task succeeded.

```python
                except Exception as e:

```

* **`except Exception as e:`**: Catches any standard error that occurred during `await handler()`, capturing the error details in variable `e`.

```python
                    if attempt == max_retries - 1:
                        async with self._lock:
                            self._results[task["id"]] = {"status": "error", "error": str(e)}

```

* **`if attempt == max_retries - 1:`**: Checks if this execution was the absolute final attempt (e.g., retry index 2 out of 3).
* **`str(e)`**: Converts the raw Python exception error message into a readable string and logs it to the results dictionary under an `"error"` status.

```python
            self._q.task_done()

```

* Confirms the item is complete, whether it ultimately succeeded or failed all retries.

---

## Level 6 — Task Dependencies (DAG)

This level executes a Directed Acyclic Graph (DAG) workflow, where certain tasks cannot run until their parent dependency tasks finish.

```python
    async def run_dag(self, tasks: dict, handler):

```

* **`tasks: dict`**: Takes a task dependency graph dictionary structure.

```python
        completed = set()
        results = {}
        lock = asyncio.Lock()

```

* **`completed = set()`**: A hash set that tracks completed task names for fast lookup checks ($O(1)$ complexity).
* **`results = {}`**: A dictionary to map task names directly to their returned outputs.
* **`lock = asyncio.Lock()`**: A local synchronization lock to safely modify the `completed` set and `results` dictionary.

```python
        async def run_task(name):

```

* **`async def run_task(name):`**: An inner coroutine function designed to manage the lifecycle of an individual task.

```python
            # Wait for all deps
            while True:
                async with lock:
                    if all(d in completed for d in tasks[name]["deps"]):
                        break
                await asyncio.sleep(0.01)

```

* **`while True:`**: A polling loop that waits for dependencies to clear.
* **`async with lock:`**: Safely guards access to the shared `completed` set.
* **`all(d in completed for d in ...)`**: Checks if **every single dependency item** listed inside this task's `"deps"` array is currently found inside the `completed` set.
* **`break`**: Breaks the polling loop if all dependencies are ready.
* **`await asyncio.sleep(0.01)`**: If dependencies are missing, it pauses execution for 10 milliseconds, yielding control back to the event loop so other tasks can run and complete.

```python
            result = await handler(tasks[name]["payload"])
            async with lock:
                results[name] = result
                completed.add(name)

```

* **`await handler(...)`**: Executes the task payload once its dependencies are cleared.
* **`results[name] = result`**: Stores the output data securely.
* **`completed.add(name)`**: Adds this task name to the `completed` set, unblocking any other tasks waiting on it.

```python
        await asyncio.gather(*[run_task(name) for name in tasks])
        return results

```

* **`[run_task(name) for name in tasks]`**: Generates an independent tracking coroutine for every task key in the dictionary.
* **`await asyncio.gather(*...)`**: Launches all of these tracking coroutines concurrently. The tasks that have no dependencies run first, while tasks with dependencies wait for their parent jobs to finish before executing.
* **`return results`**: Once all tasks complete, it returns the final results map.

In [15]:
## creating threads 

import threading
import time

def worker(name: str, delay: float):
    print(f"{name} starting")
    time.sleep(delay)
    print(f"{name} done")

# Method 1: Thread with target function
t = threading.Thread(target=worker, args=("Alice", 2), kwargs={})
t.start()
t.join()    # wait for thread to finish

# Method 2: Subclass Thread
class MyThread(threading.Thread):
    def __init__(self, name, delay):
        super().__init__()
        self.name = name
        self.delay = delay
        self.result = None

    def run(self):                  # override run(), not start()
        time.sleep(self.delay)
        self.result = f"{self.name} completed"

t = MyThread("Bob", 1)
t.start()
t.join()
print(t.result)


Alice starting
Alice done
Bob completed


In [ ]:
Class TaskQueue :
def __init__ (self) :
    

def 

#Fii buss buss
